# Pratica finale — Scegli un caso d'uso ed estendilo

Questa è l'ultima pratica del corso, ed è diversa da tutte le precedenti: non c'è un unico esercizio con un'unica soluzione. Invece, sceglierai **uno** dei casi d'uso reali che hai costruito lungo il corso e lo estenderai con un miglioramento concreto — come faresti in qualsiasi progetto finale reale.

Rivedi prima il capitolo di chiusura del manuale, [17_casos_de_uso_reales.md](../manual/17_casos_de_uso_reales_IT.md), se non l'hai ancora letto: lì trovi il riepilogo dei 7 casi d'uso (chatbot da terminale, chatbot web, trascrittore di riunioni, assistente vocale, chatbot RAG, traduttore vocale, classificazione+captioning di immagini).

Qui sotto trovi **tre sfide concrete**, ognuna basata su un caso d'uso diverso del corso. Scegline **almeno una** (puoi provarne di più se vuoi). Solo la prima porta una soluzione di esempio in `<details>` — le altre due restano aperte di proposito, proprio come accadrebbe in un progetto finale reale: non sempre c'è un'unica risposta corretta.

## Sfida 1 — Aggiungi memoria persistente al chatbot da terminale

**Caso d'uso base**: il chatbot da terminale della lezione 6 (`facebook/blenderbot-400M-distill` o `HuggingFaceTB/SmolLM2-360M-Instruct`) ricorda la conversazione solo finché il programma resta in esecuzione — al riavvio, perde tutta la cronologia.

**Sfida**: fai in modo che il chatbot salvi la conversazione in un file (ad esempio, un `.json`) alla chiusura, e la recuperi al riavvio, per poter continuare una conversazione precedente.

**Suggerimenti**:
- Usa il modulo `json` di Python per salvare una lista di messaggi (`[{"rol": "usuario", "texto": "..."}, ...]`) in un file.
- All'avvio del programma, controlla se il file esiste già (`os.path.exists(...)`) e, in tal caso, caricalo prima di iniziare il ciclo di conversazione.
- Ricorda di tagliare la cronologia alle ultime N voci prima di passarla al modello (come nel chatbot originale), anche se il file salvato su disco può contenere la cronologia completa.

In [ ]:
import json
import os

ARCHIVO_HISTORIAL = "historial_chat.json"

def cargar_historial():
    """Carica lo storico di conversazione salvato, se esiste."""
    # TODO 1: se ARCHIVO_HISTORIAL esiste, aprilo e restituisci la lista di messaggi
    #         che contiene (usando json.load). Se non esiste, restituisci una lista vuota.
    return []  # <-- sostituisci questo

def guardar_historial(historial):
    """Salva la lista di messaggi in ARCHIVO_HISTORIAL."""
    # TODO 2: scrivi `historial` in ARCHIVO_HISTORIAL come JSON (usando json.dump).
    pass


# Simulazione semplificata del ciclo del chatbot con memoria persistente:
historial = cargar_historial()
print("Messaggi precedenti caricati:", len(historial))

# In un chatbot reale, qui andrebbe il ciclo "while True" che chiede input()
# e chiama il modello, aggiungendo ogni turno a `historial`.
historial.append({"rol": "usuario", "texto": "Ciao, ti ricordi di me?"})
historial.append({"rol": "bot", "texto": "Ora sì, perché salviamo lo storico su disco."})

guardar_historial(historial)
print("Storico salvato. Riesegui questa cella per verificare che venga recuperato.")

<details>
<summary>💡 Vedi soluzione (Sfida 1)</summary>

```python
import json
import os

ARCHIVO_HISTORIAL = "historial_chat.json"

def cargar_historial():
    """Carica lo storico di conversazione salvato, se esiste."""
    if os.path.exists(ARCHIVO_HISTORIAL):
        with open(ARCHIVO_HISTORIAL, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def guardar_historial(historial):
    """Salva la lista di messaggi in ARCHIVO_HISTORIAL."""
    with open(ARCHIVO_HISTORIAL, "w", encoding="utf-8") as f:
        json.dump(historial, f, ensure_ascii=False, indent=2)


historial = cargar_historial()
print("Messaggi precedenti caricati:", len(historial))

historial.append({"rol": "usuario", "texto": "Ciao, ti ricordi di me?"})
historial.append({"rol": "bot", "texto": "Ora sì, perché salviamo lo storico su disco."})

guardar_historial(historial)
print("Storico salvato. Riesegui questa cella per verificare che venga recuperato.")
```

L'idea chiave: separare la memoria "di sessione" (la lista in RAM mentre il programma è in esecuzione) dalla memoria "persistente" (il file su disco). Un chatbot di produzione reale userebbe di norma un database invece di un file JSON semplice, ma il principio — salvare e recuperare lo stato della conversazione — è esattamente lo stesso.
</details>

## Sfida 2 — Supporta più lingue nel traduttore vocale (aperta)

**Caso d'uso base**: il traduttore della lezione 11 traduce inglese → spagnolo con `Helsinki-NLP/opus-mt-en-es`, e nella pratica di quella lezione hai già generalizzato la funzione per accettare qualsiasi lingua di destinazione supportata dalla famiglia `Helsinki-NLP/opus-mt-en-*`.

**Sfida**: costruisci un traduttore **bidirezionale** — che oltre a tradurre dall'inglese a un'altra lingua, possa tradurre da quella lingua di nuovo verso l'inglese (o anche tra due lingue diverse dall'inglese). Dovrai investigare quali modelli della famiglia `Helsinki-NLP` esistono per la direzione opposta (suggerimento: il pattern del nome cambia l'ordine dei codici lingua, ad esempio `Helsinki-NLP/opus-mt-es-en`).

**Suggerimenti per iniziare**:
- Cerca nell'Hub di Hugging Face (`https://huggingface.co/models?search=Helsinki-NLP/opus-mt`) quali coppie di lingue sono disponibili.
- Pensa a come struttureresti una funzione `traducir(texto, idioma_origen, idioma_destino)` che scelga il modello corretto in base a entrambe le lingue.
- Cosa succede se qualcuno chiede di tradurre tra due lingue per cui non esiste un modello diretto (ad esempio, francese → tedesco)? Potresti risolverlo concatenando due traduzioni (francese → inglese → tedesco)?

Questa sfida resta aperta di proposito — non c'è un'unica soluzione "corretta", così come non c'è in un progetto reale.

In [ ]:
# TODO: costruisci qui il tuo traduttore bidirezionale / multi-lingua.
# Inizia definendo la funzione e prova con almeno una coppia di lingue
# in entrambe le direzioni (ad esempio, inglese->spagnolo e spagnolo->inglese).

def traducir(texto, idioma_origen, idioma_destino):
    """
    Traduce `texto` da `idioma_origen` a `idioma_destino` usando qualche
    modello della famiglia Helsinki-NLP/opus-mt-*.
    """
    # TODO: implementa la logica di selezione del modello e traduzione.
    raise NotImplementedError("Sfida aperta: non ancora implementata")

## Sfida 3 — Aggiungi una terza classe di danno al classificatore di immagini (aperta)

**Caso d'uso base**: il classificatore della lezione 12 distingue tra due categorie di danno su aeromobili: ammaccatura (*dent*) e crepa (*crack*), usando transfer learning con VGG16.

**Sfida**: estendi il classificatore affinché riconosca una **terza** categoria — ad esempio "corrosione" (*corrosion*) o "nessun danno" (*no damage*) — trasformando il problema di classificazione binaria in classificazione multiclasse.

**Suggerimenti per iniziare**:
- Il layer di output del modello dovrebbe cambiare da `Dense(1, activation='sigmoid')` (una sola probabilità, per 2 classi) a `Dense(3, activation='softmax')` (una probabilità per ciascuna delle 3 classi).
- La funzione di perdita cambierebbe da `binary_crossentropy` a `categorical_crossentropy` (o `sparse_categorical_crossentropy`, se non fai one-hot encoding delle etichette).
- Dovresti organizzare le tue immagini in tre sottocartelle (`train/dent`, `train/crack`, `train/corrosion`, ad esempio) affinché `flow_from_directory` con `class_mode='categorical'` le riconosca automaticamente.
- Rifletti: quante immagini della nuova categoria ti servirebbero come minimo perché il modello impari qualcosa di ragionevole? Cosa succederebbe se avessi molte meno foto di "corrosione" rispetto alle altre due classi (un dataset sbilanciato)?

Anche questa sfida resta aperta — è, delle tre, quella che più assomiglia a un problema reale di dati (ottenere e organizzare immagini etichettate) oltre che a un problema di modellazione.

In [ ]:
# TODO: adatta qui l'architettura del modello della lezione 12 per
# supportare 3 classi invece di 2. Non serve addestrare il modello
# completo (non abbiamo a disposizione il dataset a 3 classi); concentrati
# sul lasciare l'architettura e la compilazione corrette per 3 categorie.

from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers

base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers:
    layer.trainable = False

# TODO: cambia l'ultimo layer Dense e la funzione di perdita per 3 classi.
model_multiclase = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    # TODO: sostituisci la riga sotto con il layer di output per 3 classi
    layers.Dense(1, activation="sigmoid"),
])

# TODO: sostituisci 'binary_crossentropy' con la funzione di perdita corretta
# per la classificazione multiclasse.
model_multiclase.compile(
    loss="binary_crossentropy",
    optimizer=optimizers.Adam(learning_rate=0.0001),
    metrics=["accuracy"],
)

model_multiclase.summary()

## Per chiudere il corso

Non ci sono altri notebook dopo questo. Se hai completato almeno una delle tre sfide, hai già fatto la cosa più importante di tutto il corso: prendere un progetto funzionante ed estenderlo per conto tuo, verificando che la tua modifica funzioni davvero (come hai imparato a fare con il bug del traduttore della lezione 11).

- Capitolo di chiusura del manuale: [17_casos_de_uso_reales.md](../manual/17_casos_de_uso_reales_IT.md)
- Notebook di lezione: [clase12_vision_lenguaje_casos_uso.ipynb](../notebooks_clase/clase12_vision_lenguaje_casos_uso_IT.ipynb)